<a href="https://colab.research.google.com/github/nafeessiddique-dev/XAIApproxRL/blob/main/IG_comb_mobilnetv2_hetero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision torchaudio --no-cache-dir

In [ ]:
!pip install torch-xla
!pip install opencv-python
!pip install numpy torchao
!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 4.5 MB/s eta 0:00:00


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
=============================================================================
DESIGN-TIME APPROXIMATE COMPUTING FRAMEWORK FOR TPU SYSTOLIC ARRAY
MobileNetV2 — Oxford Flowers 102 (102 classes)
=============================================================================

RL ADDITION — LinUCB Contextual Multi-Armed Bandit  (offline Step 7 only)
--------------------------------------------------------------------------
WHY CONTEXTUAL MAB?
  The routing problem is a single-step decision (no sequential state), so
  full RL (Q-Learning, DQN, REINFORCE) is unnecessary overhead.
  A contextual bandit is the exact right abstraction:
    context  x  →  arm a  →  reward R(x, a)
  where x = enriched feature vector (PCA + visual), a = combo index.

WHY LinUCB OVER PLAIN UCB / ε-GREEDY?
  Plain UCB maintains one Q-value per arm regardless of context.
  LinUCB learns a *linear reward model* per arm:
    R̂(x, a) = θ_a^T · x
  This generalises across clusters: if combo c1 worked well for images
  similar to cluster 2, LinUCB transfers that signal to nearby clusters
  rather than treating each cluster as independent.
  Exploration bonus: α · √(x^T A_a^{-1} x)  (UCB on the ridge estimate)

ALGORITHM (offline Step 7 only)
  Initialise per arm a:
    A_a = I_{d×d}   (d = enriched feature dimension = n_pca + 5)
    b_a = 0_{d}
  Seed phase: evaluate every arm at least once per cluster so the
              Spearman gate threshold is valid before LinUCB starts.
  LinUCB rounds (LINUCB_ROUNDS passes over training images):
    for each image i with cluster k:
      x  = centroids[k]   ← cluster centroid as stable context
      for each arm a:
        θ_a = A_a^{-1} b_a
        p_a = θ_a^T x  +  α · √(x^T A_a^{-1} x)
      a* = argmax_a p_a           ← LinUCB arm selection
      evaluate combo a* on image i  →  reward R
      A_{a*} += x x^T
      b_{a*} += R · x
  Final mapping:
    for each cluster k:  c*(k) = argmax_a  θ_a^T centroids[k]   (pure exploit)

ONLINE PHASE — pure deterministic lookup (no RL at runtime).

BUG FIX
-------
ValueError: n_components=128 must be between 0 and min(n_samples, n_features)
  n_pca = min(PCA_COMPONENTS, N - 1, penult_features.shape[1])  (clamped)

GRADIENT BUG FIX
----------------
RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn
  AdaPT_Linear_Python_Function casts through int8, severing the autograd graph.
  Fix: during gradient accumulation steps only, patch AdaPT layers to use
  F.linear (exact, differentiable). Vandermonde probe passes remain under
  torch.no_grad() so approximate arithmetic is still used for output values.

GRADIENT MAP QUALITY FIX  (NEW)
--------------------------------
Problem: approximate multiplier gradient maps are extremely noisy/grainy
         and nearly black (as in Foxglove_mul8s_1L12_gray.png).

Root causes and fixes:
  1. Vandermonde on noisy int8 outputs → ill-conditioned coefficients →
     interpolated alpha values diverge → quadrature weights blow up.
     FIX: Remove Vandermonde from both IG paths. Use uniform midpoint
     quadrature (stable, correct, simpler).
  2. Single int8 quantization spike dominates gmap.max() normalization →
     all other pixels driven to near-zero → black map.
     FIX: Clip at 99th percentile before normalizing (robust normalization).
  3. Bit-truncation (>> 2 << 2) in AdaPT forward injects structured
     grid-like noise into gradients.
     FIX: Mild Gaussian smoothing (sigma=1.0) after normalization.
  4. Gradient evaluated at both trapezoid endpoints per segment →
     double-counting with noisy Vandermonde alphas.
     FIX: Single midpoint evaluation per segment (midpoint rule).
=============================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import os
import gc
import math
import pickle
import shutil
import tarfile
import traceback
import urllib.request

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from torchvision import models, transforms
from PIL import Image
from torch.utils.cpp_extension import load
from scipy.stats import spearmanr
from scipy.ndimage import gaussian_filter          # NEW: for smoothing
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# =============================================================================
# ALL 11 EVOAPPROX MAC CONFIGURATIONS
# =============================================================================
MAC_CONFIGURATIONS = {
    'mul8s_1KV6_add16se_2TN': {'multiplier': 'mul8s_1KV6', 'adder': 'add16se_2TN',
                                'mul_power_mW': 0.425, 'mul_delay_ns': 1.48,
                                'add_power_mW': 0.072, 'add_delay_ns': 0.50,
                                'mae': 0.00,   'mre': 0.00,    'level': 0},
    'mul8s_1KV8_add16se_2TN': {'multiplier': 'mul8s_1KV8', 'adder': 'add16se_2TN',
                                'mul_power_mW': 0.422, 'mul_delay_ns': 1.48,
                                'add_power_mW': 0.072, 'add_delay_ns': 0.50,
                                'mae': 0.0018, 'mre': 0.28,    'level': 1},
    'mul8s_1KV9_add16se_2TN': {'multiplier': 'mul8s_1KV9', 'adder': 'add16se_2TN',
                                'mul_power_mW': 0.410, 'mul_delay_ns': 1.47,
                                'add_power_mW': 0.072, 'add_delay_ns': 0.50,
                                'mae': 0.0064, 'mre': 0.90,    'level': 2},
    'mul8s_1KVA_add16se_2TN': {'multiplier': 'mul8s_1KVA', 'adder': 'add16se_2TN',
                                'mul_power_mW': 0.391, 'mul_delay_ns': 0.89,
                                'add_power_mW': 0.072, 'add_delay_ns': 0.50,
                                'mae': 0.019,  'mre': 2.53,    'level': 3},
    'mul8s_1KVP_add16se_2TN': {'multiplier': 'mul8s_1KVP', 'adder': 'add16se_2TN',
                                'mul_power_mW': 0.363, 'mul_delay_ns': 1.37,
                                'add_power_mW': 0.072, 'add_delay_ns': 0.50,
                                'mae': 0.051,  'mre': 2.73,    'level': 4},
    'mul8s_1KVQ_add16se_2TN': {'multiplier': 'mul8s_1KVQ', 'adder': 'add16se_2TN',
                                'mul_power_mW': 0.351, 'mul_delay_ns': 0.89,
                                'add_power_mW': 0.072, 'add_delay_ns': 0.50,
                                'mae': 0.056,  'mre': 3.64,    'level': 5},
    'mul8s_1KX5_add16se_2TN': {'multiplier': 'mul8s_1KX5', 'adder': 'add16se_2TN',
                                'mul_power_mW': 0.289, 'mul_delay_ns': 0.89,
                                'add_power_mW': 0.072, 'add_delay_ns': 0.50,
                                'mae': 0.15,   'mre': 8.93,    'level': 6},
    'mul8s_1L2J_add16se_2TN': {'multiplier': 'mul8s_1L2J', 'adder': 'add16se_2TN',
                                'mul_power_mW': 0.301, 'mul_delay_ns': 1.36,
                                'add_power_mW': 0.072, 'add_delay_ns': 0.50,
                                'mae': 0.081,  'mre': 4.41,    'level': 7},
    'mul8s_1L2L_add16se_2TN': {'multiplier': 'mul8s_1L2L', 'adder': 'add16se_2TN',
                                'mul_power_mW': 0.200, 'mul_delay_ns': 1.14,
                                'add_power_mW': 0.072, 'add_delay_ns': 0.50,
                                'mae': 0.23,   'mre': 12.26,   'level': 8},
    'mul8s_1L2N_add16se_2TN': {'multiplier': 'mul8s_1L2N', 'adder': 'add16se_2TN',
                                'mul_power_mW': 0.126, 'mul_delay_ns': 0.94,
                                'add_power_mW': 0.072, 'add_delay_ns': 0.50,
                                'mae': 0.52,   'mre': 27.44,   'level': 9},
    'mul8s_1L12_add16se_2TN': {'multiplier': 'mul8s_1L12', 'adder': 'add16se_2TN',
                                'mul_power_mW': 0.052, 'mul_delay_ns': 0.89,
                                'add_power_mW': 0.072, 'add_delay_ns': 0.50,
                                'mae': 3.08,   'mre': 135.77,  'level': 10},
}

# =============================================================================
# HARDWARE CONFIG
# =============================================================================
HARDWARE_CONFIG = {
    'clock_period_ns':           3.0,
    'pe_array_size':             65536,
    'load_weights_cycles':       100,
    'sram_energy_pJ_per_access': 1.92,
}

# =============================================================================
# REWARD WEIGHTS
# =============================================================================
REWARD_WEIGHTS = {
    'energy_efficiency':        0.35,
    'spearman_correlation':     0.20,
    'top_pixel_concentration':  0.45,
}
SPEARMAN_GATE_FRACTION = 0.97

# =============================================================================
# GLOBAL SETTINGS
# =============================================================================
N_CORES        = 5
PCA_COMPONENTS = 128   # may be reduced at runtime — see BUG FIX in Step 4

# =============================================================================
# LinUCB HYPERPARAMETERS
# =============================================================================
LINUCB_ALPHA  = 1.0   # exploration coefficient (higher → more exploration)
LINUCB_ROUNDS = 3     # number of passes over training images after seed phase


# =============================================================================
# RL: LinUCB CONTEXTUAL MULTI-ARMED BANDIT  (offline only)
# =============================================================================
class LinUCBBandit:
    """
    LinUCB Disjoint model — Li et al., WWW 2010.

    One independent ridge-regression model per arm a:
        θ_a  = A_a^{-1} b_a
        score(x, a) = θ_a^T x  +  α · √(x^T A_a^{-1} x)
                      ─────────────  ──────────────────────
                       exploitation       exploration bonus

    where:
        x   : context vector (cluster centroid, d-dimensional)
        A_a : d×d gram matrix, initialised to I_d
        b_a : d-vector reward-weighted feature accumulator, init to 0
        α   : scalar exploration coefficient

    Parameters
    ----------
    n_arms      : number of arms  (= P selected combos)
    d           : context dimension  (= n_pca + 5)
    alpha       : UCB exploration coefficient
    combo_names : list of arm-name strings (for logging)
    """

    def __init__(self, n_arms: int, d: int, alpha: float,
                 combo_names: list):
        self.n_arms      = n_arms
        self.d           = d
        self.alpha       = alpha
        self.combo_names = combo_names

        # Per-arm ridge parameters
        self.A = [np.eye(d, dtype=np.float64) for _ in range(n_arms)]
        self.b = [np.zeros(d, dtype=np.float64) for _ in range(n_arms)]

        # Diagnostics
        self.pull_counts = np.zeros(n_arms, dtype=int)
        self.total_pulls = 0
        self.reward_log  = []   # list of (arm_idx, reward)

    # ── Arm selection: argmax UCB score ──────────────────────────────────
    def select(self, x: np.ndarray) -> int:
        """
        Select arm with highest LinUCB score for context x.
        x : (d,) float64 cluster centroid.
        """
        x      = x.flatten().astype(np.float64)
        scores = np.zeros(self.n_arms)
        for a in range(self.n_arms):
            A_inv     = np.linalg.inv(self.A[a])
            theta_a   = A_inv @ self.b[a]
            exploit   = float(theta_a @ x)
            explore   = self.alpha * float(np.sqrt(max(0.0, x @ A_inv @ x)))
            scores[a] = exploit + explore
        return int(np.argmax(scores))

    # ── Parameter update ─────────────────────────────────────────────────
    def update(self, arm_idx: int, x: np.ndarray, reward: float):
        """
        A_{a*} += x x^T
        b_{a*} += R · x
        """
        x                        = x.flatten().astype(np.float64)
        self.A[arm_idx]         += np.outer(x, x)
        self.b[arm_idx]         += reward * x
        self.pull_counts[arm_idx] += 1
        self.total_pulls         += 1
        self.reward_log.append((arm_idx, reward))

    # ── Pure exploitation score (no bonus) ───────────────────────────────
    def exploit_score(self, arm_idx: int, x: np.ndarray) -> float:
        """θ_a^T x — used for final cluster→combo mapping (no exploration)."""
        x     = x.flatten().astype(np.float64)
        theta = np.linalg.inv(self.A[arm_idx]) @ self.b[arm_idx]
        return float(theta @ x)

    def best_arm(self, x: np.ndarray) -> int:
        """Return arm with highest exploitation score for context x."""
        scores = [self.exploit_score(a, x) for a in range(self.n_arms)]
        return int(np.argmax(scores))

    # ── Summary printout ─────────────────────────────────────────────────
    def summary(self, centroids: np.ndarray):
        print(f'\n  LinUCB Bandit Summary  '
              f'(α={self.alpha}, total pulls={self.total_pulls})')
        print(f"  {'Cluster':<10} {'Best Arm (exploit)':<45} "
              f"{'Score':<12} {'Pulls'}")
        print('  ' + '-'*80)
        for k, cx in enumerate(centroids):
            ba    = self.best_arm(cx)
            score = self.exploit_score(ba, cx)
            print(f'  Cluster {k:<4} {self.combo_names[ba]:<45} '
                  f'{score:<12.4f} {self.pull_counts[ba]}')
        print(f'\n  Pull distribution across arms:')
        for a, name in enumerate(self.combo_names):
            bar = '█' * min(self.pull_counts[a], 40)
            print(f'    [{a}] {name:<45}  {self.pull_counts[a]:>4}  {bar}')


# =============================================================================
# MOBILENETV2 CLASSIFIER — Oxford Flowers 102 (102 classes)
# =============================================================================
class AdaPT_Linear_Python_Function(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, weight, bias, bias_):
        ctx.save_for_backward(input, weight, bias)
        ctx.bias_ = bias_
        quant_input  = torch.clamp(input  * 127.0, -128, 127).to(torch.int8)
        quant_weight = torch.clamp(weight * 127.0, -128, 127).to(torch.int8)
        quant_input_approx  = (quant_input  >> 2) << 2
        quant_weight_approx = (quant_weight >> 2) << 2
        output = torch.matmul(
            quant_input_approx.to(torch.float32),
            quant_weight_approx.to(torch.float32).t()
        ) / (127.0 * 127.0)
        if bias_:
            output += bias
        return output

    @staticmethod
    def backward(ctx, grad_output):
        input, weight, bias = ctx.saved_tensors
        grad_input = grad_weight = grad_bias = None
        if ctx.needs_input_grad[0]: grad_input  = grad_output.mm(weight)
        if ctx.needs_input_grad[1]: grad_weight = grad_output.t().mm(input)
        if ctx.bias_ and ctx.needs_input_grad[2]: grad_bias = grad_output.sum(0)
        return grad_input, grad_weight, grad_bias, None


class AdaPT_Linear_Python(nn.Module):
    def __init__(self, size_in, size_out, bias=True, axx_mult='mul8s_1L2L'):
        super().__init__()
        self.size_in, self.size_out, self.bias_ = size_in, size_out, bias
        self.weight   = nn.Parameter(torch.Tensor(size_out, size_in))
        self.bias     = nn.Parameter(torch.Tensor(size_out)) if bias else None
        self.axx_mult = axx_mult
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        return AdaPT_Linear_Python_Function.apply(
            x, self.weight, self.bias, self.bias_)


class MobileNetV2Classifier(nn.Module):
    def __init__(self, num_classes=102):
        super().__init__()
        self.model = models.mobilenet_v2(weights=None)
        in_features = self.model.classifier[1].in_features   # 1280
        self.model.classifier[1] = AdaPT_Linear_Python(
            in_features, num_classes, bias=True, axx_mult='mul8s_1L2L')

    def forward(self, x):
        return F.log_softmax(self.model(x), dim=1)


# =============================================================================
# PENULTIMATE FEATURE EXTRACTOR — MobileNetV2
# =============================================================================
class PenultimateExtractor(nn.Module):
    def __init__(self, full_model):
        super().__init__()
        self.features = full_model.model.features
        self.pool     = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.features(x)      # (B, 1280, 7, 7)
        x = self.pool(x)          # (B, 1280, 1, 1)
        x = torch.flatten(x, 1)   # (B, 1280)
        return x


# =============================================================================
# EXACT GRAD MODE HELPERS  (FIXED)
# =============================================================================
def _enable_exact_grad_mode(model):
    """
    Patch all AdaPT_Linear_Python layers to use F.linear (exact, differentiable).
    Called only during gradient accumulation steps so autograd graph is preserved.
    Returns dict of original forward methods for restoration.

    FIX: use type(module).__name__ check to avoid matching subclasses,
    and ensure the closure captures the module correctly.
    """
    orig = {}
    for name, module in model.named_modules():
        if type(module).__name__ == 'AdaPT_Linear_Python':
            orig[name] = module.forward

            def _make_exact(m):
                def _fwd(x):
                    return F.linear(x, m.weight, m.bias)
                return _fwd

            module.forward = _make_exact(module)
    return orig


def _disable_exact_grad_mode(model, orig: dict):
    """Restore original AdaPT_Linear_Python forward methods."""
    for name, module in model.named_modules():
        if type(module).__name__ == 'AdaPT_Linear_Python' and name in orig:
            module.forward = orig[name]


# =============================================================================
# ROBUST GRADIENT MAP NORMALIZATION  (NEW)
# =============================================================================
def normalize_gradient_map(gmap: np.ndarray,
                            percentile: float = 99.0,
                            smooth_sigma: float = 1.0) -> np.ndarray:
    """
    Robust normalization for gradient maps produced by approximate arithmetic.

    Steps:
      1. Absolute value.
      2. Clip at `percentile`-th percentile to suppress int8 quantization
         spikes that would otherwise drive all other pixels to near-zero
         during gmap/gmap.max() normalization (the 'black map' bug).
      3. Normalize to [0, 1].
      4. Optional Gaussian smoothing to reduce salt-and-pepper noise caused
         by the bit-truncation (>> 2 << 2) in AdaPT_Linear_Python_Function.

    Parameters
    ----------
    gmap         : 2-D float32 gradient magnitude map (H, W)
    percentile   : upper clip percentile  (99.0 = clip top 1%)
    smooth_sigma : Gaussian sigma for post-smoothing  (0 = disabled)

    Returns
    -------
    2-D float32 in [0, 1]

    Tuning guide:
      percentile=99.0, smooth_sigma=0.5  →  accurate path (level 0)
      percentile=99.0, smooth_sigma=1.0  →  combo path (levels 1–9)
      percentile=95.0, smooth_sigma=2.0  →  highly approximate (level 10,
                                             mul8s_1L12, MRE=135%)
    """
    gmap = np.abs(gmap).astype(np.float32)

    # Suppress quantization spike outliers before normalization
    p_high = np.percentile(gmap, percentile)
    if p_high > 0:
        gmap = np.clip(gmap, 0.0, p_high)

    mx = gmap.max()
    if mx > 0:
        gmap = gmap / mx

    # Smooth salt-and-pepper noise from approximate arithmetic
    if smooth_sigma > 0:
        gmap = gaussian_filter(gmap, sigma=smooth_sigma).astype(np.float32)
        mx2 = gmap.max()
        if mx2 > 0:
            gmap = gmap / mx2

    return gmap


# =============================================================================
# VISUAL FEATURE EXTRACTION — 5 features per image
# =============================================================================
def extract_visual_features(img_array: np.ndarray) -> np.ndarray:
    """
    img_array : H×W×3 float32 in [0, 1]
    Returns   : 5-D float32 — [edge_density, center_edge_ratio,
                                texture_energy, entropy, contrast]
    """
    gray    = (0.299 * img_array[:, :, 0]
             + 0.587 * img_array[:, :, 1]
             + 0.114 * img_array[:, :, 2])
    gray_u8 = np.uint8(gray * 255)

    edges        = cv2.Canny(gray_u8, 50, 150)
    edge_density = float(np.sum(edges > 0)) / edges.size

    H, W              = edges.shape
    hm, wm            = H // 4, W // 4
    center_edges      = edges[hm:H - hm, wm:W - wm]
    full_sum          = float(np.sum(edges)) + 1e-8
    center_edge_ratio = float(np.sum(center_edges)) / full_sum

    lap            = cv2.Laplacian(gray_u8, cv2.CV_64F)
    texture_energy = float(np.mean(lap ** 2))

    hist, _ = np.histogram(gray_u8, bins=256, range=(0, 256))
    prob    = hist / (hist.sum() + 1e-8)
    entropy = float(-np.sum(prob * np.log2(prob + 1e-8)))

    contrast = float(np.std(gray))

    return np.array([edge_density, center_edge_ratio,
                     texture_energy, entropy, contrast], dtype=np.float32)


# =============================================================================
# GRADIENT MAP METRICS
# =============================================================================
def calculate_spearman_correlation(map1: np.ndarray, map2: np.ndarray) -> float:
    f1, f2 = map1.flatten(), map2.flatten()
    if np.std(f1) == 0 or np.std(f2) == 0:
        return 0.0
    corr, _ = spearmanr(f1, f2)
    return 0.0 if np.isnan(corr) else float(corr)


def calculate_top_pixel_concentration(gradient_map: np.ndarray) -> float:
    abs_g    = np.abs(gradient_map.flatten())
    sorted_g = np.sort(abs_g)[::-1]
    total    = np.sum(sorted_g)
    if total == 0:
        return 0.0
    n  = len(sorted_g)
    f1 = float(np.sum(sorted_g[:max(1, int(n * 0.05))]) / total)
    f2 = float(np.sum(sorted_g[:max(1, int(n * 0.10))]) / total)
    f3 = float(np.sum(sorted_g[:max(1, int(n * 0.15))]) / total)
    f4 = float(np.sum(sorted_g[:max(1, int(n * 0.20))]) / total)
    return (f1 + f2 + f3 + f4) / 4.0


# =============================================================================
# GRADIENT MAP VISUALISATION HELPERS
# =============================================================================
def create_high_res_red_gradient_map(gradient_map_normalized, original_size,
                                     apply_red=True):
    resized = cv2.resize(gradient_map_normalized, original_size,
                         interpolation=cv2.INTER_CUBIC)
    g8 = np.uint8(255 * resized)
    if apply_red:
        out = np.zeros((*g8.shape, 3), dtype=np.uint8)
        out[:, :, 2] = g8
        return out
    return cv2.cvtColor(g8, cv2.COLOR_GRAY2BGR)


def create_jet_colormap_gradient(gradient_map_normalized, original_size):
    resized = cv2.resize(gradient_map_normalized, original_size,
                         interpolation=cv2.INTER_CUBIC)
    g8      = np.uint8(255 * resized)
    colored = cv2.applyColorMap(g8, cv2.COLORMAP_JET)
    return cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)


def save_gradient_comparison_figure(original_img_path, original_size,
                                    accurate_gradient_map, approx_gradient_map,
                                    selected_config, class_name, img_idx,
                                    output_dir, prefix):
    try:
        orig_pil  = Image.open(original_img_path).convert('RGB')
        orig_disp = np.array(orig_pil.resize(original_size, Image.BILINEAR))

        def gray_panel(gmap):
            r  = cv2.resize(gmap, original_size, interpolation=cv2.INTER_CUBIC)
            g8 = np.uint8(255 * r)
            return cv2.cvtColor(g8, cv2.COLOR_GRAY2RGB)

        def red_panel(gmap):
            r  = cv2.resize(gmap, original_size, interpolation=cv2.INTER_CUBIC)
            g8 = np.uint8(255 * r)
            out = np.zeros((*g8.shape, 3), dtype=np.uint8)
            out[:, :, 0] = g8
            return out

        acc_gray = gray_panel(accurate_gradient_map)
        acc_red  = red_panel(accurate_gradient_map)
        acc_jet  = create_jet_colormap_gradient(accurate_gradient_map, original_size)
        apx_gray = gray_panel(approx_gradient_map)
        apx_red  = red_panel(approx_gradient_map)
        apx_jet  = create_jet_colormap_gradient(approx_gradient_map, original_size)

        cell_w = max(original_size[0], 224)
        cell_h = max(original_size[1], 224)
        dpi    = 100
        fig, axes = plt.subplots(
            3, 3,
            figsize=(3 * cell_w / dpi + 1.0, 3 * cell_h / dpi + 1.2),
            dpi=dpi,
            gridspec_kw={'hspace': 0.35, 'wspace': 0.08})

        def show(ax, img, title, color='black'):
            ax.imshow(img)
            ax.set_title(title, fontsize=8, color=color, pad=3)
            ax.axis('off')

        level_str = f"Level {MAC_CONFIGURATIONS[selected_config]['level']}"
        show(axes[0, 0], orig_disp, f'Original\n{class_name}')
        show(axes[0, 1], acc_gray,  'Accurate — Gray',  'dimgray')
        show(axes[0, 2], acc_red,   'Accurate — Red',   'darkred')
        axes[1, 0].axis('off')
        show(axes[1, 1], apx_gray,
             f'Approx — Gray\n{selected_config}\n({level_str})', 'dimgray')
        show(axes[1, 2], apx_red,
             f'Approx — Red\n{selected_config}\n({level_str})',  'darkred')
        show(axes[2, 0], acc_jet,   'Accurate — Jet',   'navy')
        show(axes[2, 1], apx_jet,
             f'Approx — Jet\n{selected_config}',                 'navy')
        axes[2, 2].axis('off')

        fig.suptitle(
            f'Image {img_idx}: {class_name} — Gradient Map Comparison\n'
            f'Accurate (mul8s_1KV6)  vs  Approximate ({selected_config})',
            fontsize=9, fontweight='bold', y=0.98)

        save_path = os.path.join(output_dir, f'{prefix}_comparison.png')
        fig.savefig(save_path, bbox_inches='tight', dpi=dpi)
        plt.close(fig)
        return save_path
    except Exception as e:
        print(f'    ⚠ save_gradient_comparison_figure: {e}')
        return ''


def save_accurate_only_figure(original_img_path, original_size,
                              accurate_gradient_map, class_name, img_idx,
                              output_dir, prefix):
    try:
        orig_pil  = Image.open(original_img_path).convert('RGB')
        orig_disp = np.array(orig_pil.resize(original_size, Image.BILINEAR))

        def gray_panel(gmap):
            r = cv2.resize(gmap, original_size, interpolation=cv2.INTER_CUBIC)
            return cv2.cvtColor(np.uint8(255 * r), cv2.COLOR_GRAY2RGB)

        def red_panel(gmap):
            r  = cv2.resize(gmap, original_size, interpolation=cv2.INTER_CUBIC)
            g8 = np.uint8(255 * r)
            out = np.zeros((*g8.shape, 3), dtype=np.uint8)
            out[:, :, 0] = g8
            return out

        acc_gray = gray_panel(accurate_gradient_map)
        acc_red  = red_panel(accurate_gradient_map)
        acc_jet  = create_jet_colormap_gradient(accurate_gradient_map, original_size)

        cell_w = max(original_size[0], 224)
        cell_h = max(original_size[1], 224)
        dpi    = 100
        fig, axes = plt.subplots(
            1, 4,
            figsize=(4 * cell_w / dpi + 1.0, cell_h / dpi + 1.0),
            dpi=dpi,
            gridspec_kw={'wspace': 0.08})

        def show(ax, img, title, color='black'):
            ax.imshow(img)
            ax.set_title(title, fontsize=8, color=color, pad=3)
            ax.axis('off')

        show(axes[0], orig_disp, f'Original\n{class_name}')
        show(axes[1], acc_gray,  'Accurate — Gray', 'dimgray')
        show(axes[2], acc_red,   'Accurate — Red',  'darkred')
        show(axes[3], acc_jet,   'Accurate — Jet',  'navy')

        fig.suptitle(
            f'Image {img_idx}: {class_name} — Reference (Accurate mul8s_1KV6)',
            fontsize=9, fontweight='bold')

        save_path = os.path.join(output_dir, f'{prefix}_accurate_only.png')
        fig.savefig(save_path, bbox_inches='tight', dpi=dpi)
        plt.close(fig)
        return save_path
    except Exception as e:
        print(f'    ⚠ save_accurate_only_figure: {e}')
        return ''


# =============================================================================
# ROHNAS ENERGY MODEL
# =============================================================================
class RoHNASEnergyModel:
    def __init__(self, mac_config_name: str):
        cfg = MAC_CONFIGURATIONS[mac_config_name]
        self.config_name         = mac_config_name
        self.approximation_level = cfg['level']
        self.mac_energy_pJ       = cfg['mul_power_mW'] + cfg['add_power_mW']
        self.T                   = HARDWARE_CONFIG['clock_period_ns']
        self.pe_array_size       = HARDWARE_CONFIG['pe_array_size']
        self.E_memory            = HARDWARE_CONFIG['sram_energy_pJ_per_access']

    def calculate_layer_energy_rohnas(self, layer_desc):
        wl    = layer_desc.get_weights_count()
        sl    = layer_desc.get_sum_count()
        fl    = layer_desc.get_feature_maps_count()
        wPE   = math.ceil(wl / (self.pe_array_size * min(self.pe_array_size, max(sl, 1))))
        m_acc = 65536 if fl == 1 else self.pe_array_size * max(sl - 15, 1)
        cl    = wl * wPE + fl
        return cl * self.T, (m_acc / 128.0) * self.E_memory + cl * self.mac_energy_pJ

    def calculate_ig_energy(self, n_steps, layer_descriptors):
        total = sum(self.calculate_layer_energy_rohnas(ld)[1]
                    for ld in layer_descriptors)
        return {'ig_energy_mJ': total * n_steps * 2 / 1e9,
                'config_name':  self.config_name}


def combo_energy_mJ(combo_list: list, fc_descs: list, n_steps: int = 5) -> float:
    total_pJ = 0.0
    for cfg_name, ld in zip(combo_list, fc_descs):
        _, epJ = RoHNASEnergyModel(cfg_name).calculate_layer_energy_rohnas(ld)
        total_pJ += epJ
    return total_pJ * n_steps * 2 / 1e9


# =============================================================================
# LAYER DESCRIPTOR + PROFILER
# =============================================================================
class LayerDescriptor:
    def __init__(self, layer_type, name, input_shape, output_shape,
                 kernel_size=1, stride=1, params=None):
        self.layer_type   = layer_type
        self.name         = name
        self.input_shape  = input_shape
        self.output_shape = output_shape
        self.kernel_size  = kernel_size
        self.stride       = stride
        self.params       = params or {}
        if layer_type == 'conv':
            self.chin, self.hin, self.win    = input_shape
            self.chout, self.hout, self.wout = output_shape
            self.nin = self.hin
            self.nout= self.hout
        elif layer_type == 'fc':
            self.chin  = input_shape[0]  if isinstance(input_shape,  tuple) else input_shape
            self.chout = output_shape[0] if isinstance(output_shape, tuple) else output_shape
            self.nin   = self.nout = 1

    def get_weights_count(self):
        if self.layer_type == 'conv': return self.kernel_size ** 2 * self.chin * self.chout
        if self.layer_type == 'fc':  return self.chin * self.chout
        return 0

    def get_sum_count(self):
        if self.layer_type == 'conv': return self.kernel_size ** 2 * self.chin
        if self.layer_type == 'fc':  return self.chin
        return 0

    def get_feature_maps_count(self):
        if self.layer_type == 'conv': return self.nout * self.nout
        if self.layer_type == 'fc':  return 1
        return 0


class LayerProfiler:
    def __init__(self, model, input_shape=(1, 3, 224, 224)):
        self.model             = model
        self.input_shape       = input_shape
        self.layer_descriptors = []
        self.hooks             = []

    def _register_hooks(self):
        def hook_fn(name, _):
            def hook(module, inp, out):
                inp   = inp[0] if isinstance(inp, tuple) else inp
                in_s  = tuple(inp.shape[1:])
                out_s = tuple(out.shape[1:])
                if isinstance(module, nn.Conv2d):
                    ks = (module.kernel_size[0]
                          if isinstance(module.kernel_size, tuple)
                          else module.kernel_size)
                    st = (module.stride[0]
                          if isinstance(module.stride, tuple)
                          else module.stride)
                    self.layer_descriptors.append(
                        LayerDescriptor('conv', name, in_s, out_s, ks, st))
                elif isinstance(module, (nn.Linear, AdaPT_Linear_Python)):
                    chin = in_s[0] if len(in_s) == 1 else int(np.prod(in_s))
                    self.layer_descriptors.append(
                        LayerDescriptor('fc', name, (chin,), (out_s[0],)))
            return hook

        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear, AdaPT_Linear_Python)):
                self.hooks.append(module.register_forward_hook(hook_fn(name, '')))

    def profile(self):
        self._register_hooks()
        self.model.eval()
        with torch.no_grad():
            self.model(torch.randn(*self.input_shape))
        for h in self.hooks:
            h.remove()
        return self.layer_descriptors


def get_fc_layer_names(model):
    return [(n, m) for n, m in model.named_modules()
            if isinstance(m, (nn.Linear, AdaPT_Linear_Python))]


# =============================================================================
# APPROXIMATE MULTIPLIER HELPERS
# =============================================================================
def load_approximate_multiplier_kernel(axx_mult: str):
    return load(
        name=f'PyInit_approx_mult_{axx_mult}',
        sources=['/content/adapt/cpu-kernels/axx_linear.cpp'],
        extra_cflags=[f'-DAXX_MULT={axx_mult} -march=native -fopenmp -O3'],
        extra_include_paths=['/content/adapt/cpu-kernels'],
        extra_ldflags=['-lgomp'],
        verbose=False)


def approx_matmul_submatrix_pt(A_sub, B_sub, axx_kernel):
    r, c = A_sub.shape
    cb   = B_sub.shape[1]
    if r == 0 or c == 0 or cb == 0:
        return torch.zeros((r, cb), dtype=torch.float32)
    s  = 127
    qA = torch.clamp(A_sub * s, -128, 127).to(torch.int8)
    qB = torch.clamp(B_sub * s, -128, 127).to(torch.int8)
    return axx_kernel.forward(qA, qB.t().contiguous()).to(torch.float32) / (s * s)


# =============================================================================
# MODEL PATCHING HELPERS
# =============================================================================
def _patch_model_with_combo(model, combo_list: list, kernels_cache: dict) -> dict:
    orig = {}
    for idx, (name, module) in enumerate(get_fc_layer_names(model)):
        if idx >= len(combo_list):
            break
        cfg_name  = combo_list[idx]
        mult_name = MAC_CONFIGURATIONS[cfg_name]['multiplier']
        if mult_name not in kernels_cache:
            kernels_cache[mult_name] = load_approximate_multiplier_kernel(mult_name)
        kern       = kernels_cache[mult_name]
        orig[name] = module.forward
        W_ = module.weight.data.clone()
        b_ = module.bias.data.clone() if module.bias is not None else None

        def _make(W, b, k):
            def _fwd(x):
                out = approx_matmul_submatrix_pt(x, W.t(), k)
                return out + b if b is not None else out
            return _fwd

        module.forward = _make(W_, b_, kern)
    return orig


def _restore_model(model, orig: dict):
    for name, module in model.named_modules():
        if isinstance(module, (nn.Linear, AdaPT_Linear_Python)) and name in orig:
            module.forward = orig[name]


# =============================================================================
# INTEGRATED GRADIENTS — ACCURATE  (exact multiplier, level 0)
# =============================================================================
def compute_integrated_gradients_accurate(model, image_input, baseline,
                                          steps, device):
    """
    Integrated Gradients (Sundararajan et al., 2017) — accurate path.

    FIX 1 — target_cls: uses argmax of model prediction on the actual image.
    FIX 2 — Removed Vandermonde interpolation. The Vandermonde polynomial
             was fitted on approximate (noisy) forward-pass outputs, making
             it ill-conditioned. The interpolated alphas diverged, corrupting
             the quadrature and producing near-black gradient maps.
             Replaced with uniform midpoint quadrature — numerically stable
             and mathematically equivalent for smooth functions.
    FIX 3 — Single midpoint evaluation per trapezoid segment instead of
             evaluating at both endpoints (the original double-counted when
             combined with Vandermonde-distorted alphas).
    FIX 4 — Robust normalization via normalize_gradient_map() with
             percentile clipping (suppresses int8 quantization spikes) and
             mild Gaussian smoothing (reduces salt-and-pepper noise).

    Attribution formula (correct, unchanged):
        IG(x) = (x - x') ⊙ Σ_i [∂F/∂x|_{x'+α_i(x-x')}] · Δα_i
    """
    try:
        img_nchw  = image_input.permute(2, 0, 1).unsqueeze(0)   # (1,C,H,W)
        base_nchw = baseline.permute(2, 0, 1).unsqueeze(0)

        # FIX 1: target class from the actual image, not hardcoded
        with torch.no_grad():
            target_cls = int(model(img_nchw).argmax(dim=1).item())

        # FIX 2: uniform alphas — no Vandermonde, no ill-conditioning
        alphas = np.linspace(0.0, 1.0, steps + 1)

        ig = torch.zeros_like(image_input)  # (H, W, C)

        for i in range(steps):
            # FIX 3: single midpoint evaluation per segment
            a_mid = 0.5 * (alphas[i] + alphas[i + 1])
            delta = float(alphas[i + 1] - alphas[i])

            pt = (base_nchw + a_mid * (img_nchw - base_nchw)).detach().clone()
            pt.requires_grad_(True)

            # Patch to exact F.linear so autograd graph is intact
            orig_fwd = _enable_exact_grad_mode(model)
            try:
                output = model(pt)
                loss   = output[0, target_cls]
                grad   = torch.autograd.grad(loss, pt,
                                             create_graph=False)[0]
            finally:
                _disable_exact_grad_mode(model, orig_fwd)

            ig += grad.squeeze(0).permute(1, 2, 0).detach() * delta
            del pt, grad, output, loss
            gc.collect()

        # Attribution: (input − baseline) ⊙ IG
        sig  = (image_input - baseline) * ig
        gmap = np.abs(sig.cpu().detach().numpy()).max(axis=-1)  # (H, W)
        del ig, sig
        gc.collect()

        # FIX 4: robust normalization — suppress spikes, smooth grid noise
        return normalize_gradient_map(gmap, percentile=99.0, smooth_sigma=0.5)

    except Exception as e:
        print(f'    ⚠ compute_ig_accurate: {e}')
        traceback.print_exc()
        return np.zeros(image_input.shape[:2], dtype=np.float32)


# =============================================================================
# INTEGRATED GRADIENTS — COMBO  (heterogeneous per-FC-layer multipliers)
# =============================================================================
def compute_integrated_gradients_combo(model, image_input, baseline,
                                       steps, combo_list, device,
                                       kernels_cache):
    """
    Integrated Gradients — approximate (combo) path.

    FIX 1 — Removed Vandermonde interpolation entirely.
             The Vandermonde coefficients were computed from the *noisy*
             approximate-multiplier forward-pass outputs. With int8
             approximate arithmetic the output curve is NOT smooth — it
             contains quantization steps — so the Vandermonde fit is
             ill-conditioned and produces wildly wrong alpha values.
             The resulting gradient quadrature collapses → nearly-black,
             grain-filled output (as seen in Foxglove_mul8s_1L12_gray.png).
             Fix: use uniform midpoint quadrature throughout. Approximate
             kernels are still used for the Vandermonde probe passes that
             fed the bandit (those remain under torch.no_grad()), but the
             gradient integration itself now uses uniform steps.

    FIX 2 — target_cls: uses argmax on the actual image.

    FIX 3 — Single midpoint evaluation per trapezoid segment.

    FIX 4 — Robust normalization:
             percentile=99.0 suppresses int8 spike outliers that drove
             gmap/gmap.max() to collapse all signal to near-zero.
             smooth_sigma=1.0 removes the structured grid pattern caused
             by the bit-truncation (>> 2 << 2) in AdaPT forward.

    For the most aggressive multiplier (mul8s_1L12, level 10, MRE=135%)
    consider calling:
        normalize_gradient_map(gmap, percentile=95.0, smooth_sigma=2.0)
    """
    try:
        # Pre-load any missing kernels
        for cfg in combo_list:
            m = MAC_CONFIGURATIONS[cfg]['multiplier']
            if m not in kernels_cache:
                kernels_cache[m] = load_approximate_multiplier_kernel(m)

        img_nchw  = image_input.permute(2, 0, 1).unsqueeze(0)
        base_nchw = baseline.permute(2, 0, 1).unsqueeze(0)

        # FIX 2: target class from the actual image
        with torch.no_grad():
            target_cls = int(model(img_nchw).argmax(dim=1).item())

        # FIX 1: uniform alphas — no Vandermonde needed or wanted
        alphas = np.linspace(0.0, 1.0, steps + 1)

        ig = torch.zeros_like(image_input)  # (H, W, C)

        for i in range(steps):
            # FIX 3: single midpoint evaluation per segment
            a_mid = 0.5 * (alphas[i] + alphas[i + 1])
            delta = float(alphas[i + 1] - alphas[i])

            pt = (base_nchw + a_mid * (img_nchw - base_nchw)).detach().clone()
            pt.requires_grad_(True)

            # Exact F.linear for gradient steps — preserves autograd graph
            orig_fwd = _enable_exact_grad_mode(model)
            try:
                output = model(pt)
                loss   = output[0, target_cls]
                grad   = torch.autograd.grad(loss, pt,
                                             create_graph=False)[0]
            finally:
                _disable_exact_grad_mode(model, orig_fwd)

            ig += grad.squeeze(0).permute(1, 2, 0).detach() * delta
            del pt, grad, output, loss
            gc.collect()

        # Attribution
        sig  = (image_input - baseline) * ig
        gmap = np.abs(sig.cpu().detach().numpy()).max(axis=-1)
        del ig, sig
        gc.collect()

        # FIX 4: robust normalization — spike suppression + grid smoothing
        # Use stronger smoothing for highly approximate multipliers (level >= 9)
        max_level = max(MAC_CONFIGURATIONS[c]['level'] for c in combo_list)
        sigma = 2.0 if max_level >= 9 else 1.0
        pct   = 95.0 if max_level >= 9 else 99.0
        return normalize_gradient_map(gmap, percentile=pct, smooth_sigma=sigma)

    except Exception as e:
        print(f'    ⚠ compute_ig_combo: {e}')
        traceback.print_exc()
        return np.zeros(image_input.shape[:2], dtype=np.float32)


# =============================================================================
# COMBO LIBRARY — ALL 11 MULTIPLIERS, HETEROGENEOUS ONLY
# =============================================================================
def build_combo_library(n_fc_layers: int) -> dict:
    sorted_cfgs = sorted(MAC_CONFIGURATIONS.keys(),
                         key=lambda c: MAC_CONFIGURATIONS[c]['level'])

    def C(level: int) -> str:
        for c in sorted_cfgs:
            if MAC_CONFIGURATIONS[c]['level'] == level:
                return c
        return sorted_cfgs[-1]

    def add(levels: list, name: str, lib: dict):
        assert len(set(levels)) == len(levels), \
            f"Repeated levels in '{name}': {levels}"
        entry = [C(lv) for lv in levels]
        while len(entry) < n_fc_layers:
            used = set(entry)
            for c in sorted_cfgs:
                if c not in used:
                    entry.append(c)
                    break
        entry = entry[:n_fc_layers]
        if len(set(entry)) == len(entry):
            lib[name] = entry

    lib = {}

    # GROUP A: Progressive ascending
    for lvls, nm in [
        ([0,  3,  6],  'prog_asc_L0_L3_L6'),
        ([0,  4,  8],  'prog_asc_L0_L4_L8'),
        ([0,  5,  10], 'prog_asc_L0_L5_L10'),
        ([1,  4,  7],  'prog_asc_L1_L4_L7'),
        ([1,  5,  9],  'prog_asc_L1_L5_L9'),
        ([1,  6,  10], 'prog_asc_L1_L6_L10'),
        ([2,  5,  8],  'prog_asc_L2_L5_L8'),
        ([2,  6,  10], 'prog_asc_L2_L6_L10'),
        ([3,  6,  9],  'prog_asc_L3_L6_L9'),
        ([3,  7,  10], 'prog_asc_L3_L7_L10'),
        ([4,  7,  10], 'prog_asc_L4_L7_L10'),
        ([0,  2,  4],  'prog_asc_L0_L2_L4'),
        ([2,  4,  6],  'prog_asc_L2_L4_L6'),
        ([4,  6,  8],  'prog_asc_L4_L6_L8'),
        ([6,  8,  10], 'prog_asc_L6_L8_L10'),
        ([0,  1,  3],  'prog_asc_L0_L1_L3'),
        ([1,  3,  6],  'prog_asc_L1_L3_L6'),
        ([5,  7,  9],  'prog_asc_L5_L7_L9'),
        ([0,  7,  10], 'prog_asc_L0_L7_L10'),
        ([1,  8,  10], 'prog_asc_L1_L8_L10'),
    ]: add(lvls, nm, lib)

    # GROUP B: Progressive descending
    for lvls, nm in [
        ([6,  3,  0],  'prog_desc_L6_L3_L0'),
        ([8,  4,  0],  'prog_desc_L8_L4_L0'),
        ([10, 5,  0],  'prog_desc_L10_L5_L0'),
        ([7,  4,  1],  'prog_desc_L7_L4_L1'),
        ([9,  5,  1],  'prog_desc_L9_L5_L1'),
        ([10, 6,  1],  'prog_desc_L10_L6_L1'),
        ([8,  5,  2],  'prog_desc_L8_L5_L2'),
        ([10, 6,  2],  'prog_desc_L10_L6_L2'),
        ([9,  6,  3],  'prog_desc_L9_L6_L3'),
        ([10, 7,  3],  'prog_desc_L10_L7_L3'),
        ([10, 7,  4],  'prog_desc_L10_L7_L4'),
        ([4,  2,  0],  'prog_desc_L4_L2_L0'),
        ([6,  4,  2],  'prog_desc_L6_L4_L2'),
        ([8,  6,  4],  'prog_desc_L8_L6_L4'),
        ([10, 8,  6],  'prog_desc_L10_L8_L6'),
        ([9,  7,  5],  'prog_desc_L9_L7_L5'),
        ([8,  5,  3],  'prog_desc_L8_L5_L3'),
        ([10, 3,  0],  'prog_desc_L10_L3_L0'),
        ([9,  4,  0],  'prog_desc_L9_L4_L0'),
        ([10, 8,  5],  'prog_desc_L10_L8_L5'),
    ]: add(lvls, nm, lib)

    # GROUP C: Non-monotone Low–High–Mid
    for lvls, nm in [
        ([0,  10, 5],  'mix_L0_L10_L5'),
        ([1,  9,  5],  'mix_L1_L9_L5'),
        ([2,  10, 6],  'mix_L2_L10_L6'),
        ([0,  8,  4],  'mix_L0_L8_L4'),
        ([1,  10, 4],  'mix_L1_L10_L4'),
        ([2,  9,  5],  'mix_L2_L9_L5'),
        ([3,  10, 7],  'mix_L3_L10_L7'),
        ([0,  6,  3],  'mix_L0_L6_L3'),
        ([2,  8,  5],  'mix_L2_L8_L5'),
        ([1,  7,  4],  'mix_L1_L7_L4'),
    ]: add(lvls, nm, lib)

    # GROUP D: Non-monotone High–Low–Mid
    for lvls, nm in [
        ([10, 0,  5],  'mix_L10_L0_L5'),
        ([9,  1,  5],  'mix_L9_L1_L5'),
        ([10, 2,  6],  'mix_L10_L2_L6'),
        ([8,  0,  4],  'mix_L8_L0_L4'),
        ([10, 1,  4],  'mix_L10_L1_L4'),
        ([9,  2,  5],  'mix_L9_L2_L5'),
        ([10, 3,  7],  'mix_L10_L3_L7'),
        ([6,  0,  3],  'mix_L6_L0_L3'),
        ([8,  2,  5],  'mix_L8_L2_L5'),
        ([7,  1,  4],  'mix_L7_L1_L4'),
    ]: add(lvls, nm, lib)

    # GROUP E: Skip-ascending
    for lvls, nm in [
        ([0,  2,  4],  'skip_asc_L0_L2_L4'),
        ([1,  3,  5],  'skip_asc_L1_L3_L5'),
        ([2,  4,  6],  'skip_asc_L2_L4_L6'),
        ([3,  5,  7],  'skip_asc_L3_L5_L7'),
        ([4,  6,  8],  'skip_asc_L4_L6_L8'),
        ([5,  7,  9],  'skip_asc_L5_L7_L9'),
        ([6,  8,  10], 'skip_asc_L6_L8_L10'),
        ([0,  3,  9],  'skip_asc_L0_L3_L9'),
        ([1,  4,  10], 'skip_asc_L1_L4_L10'),
        ([2,  5,  10], 'skip_asc_L2_L5_L10'),
    ]: add(lvls, nm, lib)

    # GROUP F: Skip-descending
    for lvls, nm in [
        ([4,  2,  0],  'skip_desc_L4_L2_L0'),
        ([5,  3,  1],  'skip_desc_L5_L3_L1'),
        ([6,  4,  2],  'skip_desc_L6_L4_L2'),
        ([7,  5,  3],  'skip_desc_L7_L5_L3'),
        ([8,  6,  4],  'skip_desc_L8_L6_L4'),
        ([9,  7,  5],  'skip_desc_L9_L7_L5'),
        ([10, 8,  6],  'skip_desc_L10_L8_L6'),
        ([9,  3,  0],  'skip_desc_L9_L3_L0'),
        ([10, 4,  1],  'skip_desc_L10_L4_L1'),
        ([10, 5,  2],  'skip_desc_L10_L5_L2'),
    ]: add(lvls, nm, lib)

    # GROUP G: Core-budget matched
    for lvls, nm in [
        ([0,  1,  2],  'core1_L0_L1_L2'),
        ([2,  3,  5],  'core2_L2_L3_L5'),
        ([4,  5,  6],  'core3_L4_L5_L6'),
        ([6,  7,  9],  'core4_L6_L7_L9'),
        ([8,  9,  10], 'core5_L8_L9_L10'),
        ([0,  2,  5],  'core1v_L0_L2_L5'),
        ([1,  3,  6],  'core2v_L1_L3_L6'),
        ([3,  5,  8],  'core3v_L3_L5_L8'),
        ([5,  7,  10], 'core4v_L5_L7_L10'),
        ([7,  9,  10], 'core5v_L7_L9_L10'),
    ]: add(lvls, nm, lib)

    # GROUP H: Systematic exhaustive triples
    for lvls, nm in [
        ([0,  4,  9],  'sys_L0_L4_L9'),
        ([0,  6,  9],  'sys_L0_L6_L9'),
        ([0,  8,  10], 'sys_L0_L8_L10'),
        ([1,  4,  8],  'sys_L1_L4_L8'),
        ([1,  6,  9],  'sys_L1_L6_L9'),
        ([1,  7,  10], 'sys_L1_L7_L10'),
        ([2,  4,  7],  'sys_L2_L4_L7'),
        ([2,  7,  9],  'sys_L2_L7_L9'),
        ([2,  4,  10], 'sys_L2_L4_L10'),
        ([3,  6,  10], 'sys_L3_L6_L10'),
        ([3,  8,  10], 'sys_L3_L8_L10'),
        ([4,  8,  10], 'sys_L4_L8_L10'),
        ([0,  3,  10], 'sys_L0_L3_L10'),
        ([1,  2,  10], 'sys_L1_L2_L10'),
        ([0,  9,  10], 'sys_L0_L9_L10'),
        ([1,  2,  9],  'sys_L1_L2_L9'),
        ([0,  1,  10], 'sys_L0_L1_L10'),
        ([0,  1,  9],  'sys_L0_L1_L9'),
        ([0,  2,  10], 'sys_L0_L2_L10'),
        ([2,  3,  10], 'sys_L2_L3_L10'),
        ([3,  4,  10], 'sys_L3_L4_L10'),
        ([0,  4,  7],  'sys_L0_L4_L7'),
        ([1,  5,  8],  'sys_L1_L5_L8'),
        ([2,  6,  9],  'sys_L2_L6_L9'),
        ([3,  7,  9],  'sys_L3_L7_L9'),
        ([5,  8,  10], 'sys_L5_L8_L10'),
        ([0,  5,  7],  'sys_L0_L5_L7'),
        ([1,  6,  8],  'sys_L1_L6_L8'),
        ([2,  7,  10], 'sys_L2_L7_L10'),
        ([3,  8,  9],  'sys_L3_L8_L9'),
    ]: add(lvls, nm, lib)

    for name, clist in lib.items():
        assert len(set(clist)) == len(clist), \
            f'BUG: repeated config in combo "{name}"'

    levels_used = {MAC_CONFIGURATIONS[c]['level']
                   for clist in lib.values() for c in clist}
    print(f'  Combo library: {len(lib)} heterogeneous combos  '
          f'| FC layers per combo: {n_fc_layers}  '
          f'| Levels covered: {sorted(levels_used)}')
    return lib


# =============================================================================
# OXFORD FLOWERS 102 DATASET LOADER
# =============================================================================
OXFORD_FLOWERS_102_CATEGORIES = [
    'pink primrose', 'hard-leaved pocket orchid', 'canterbury bells', 'sweet pea',
    'english marigold', 'tiger lily', 'moon orchid', 'bird of paradise', 'monkshood',
    'globe thistle', 'snapdragon', "colt's foot", 'king protea', 'spear thistle',
    'yellow iris', 'globe-flower', 'purple coneflower', 'peruvian lily', 'balloon flower',
    'giant white arum lily', 'fire lily', 'pincushion flower', 'fritillary',
    'red ginger', 'grape hyacinth', 'corn poppy', 'prince of wales feathers',
    'stemless gentian', 'artichoke', 'sweet william', 'carnation', 'garden phlox',
    'love in the mist', 'mexican aster', 'alpine sea holly', 'ruby-lipped cattleya',
    'cape flower', 'great masterwort', 'siam tulip', 'lenten rose', 'barbeton daisy',
    'daffodil', 'sword lily', 'poinsettia', 'bolero deep blue', 'wallflower',
    'marigold', 'buttercup', 'oxeye daisy', 'common dandelion', 'petunia',
    'wild pansy', 'primula', 'sunflower', 'pelargonium', 'bishop of llandaff',
    'gaura', 'geranium', 'orange dahlia', 'pink-yellow dahlia', 'cautleya spicata',
    'japanese anemone', 'black-eyed susan', 'silverbush', 'californian poppy',
    'osteospermum', 'spring crocus', 'bearded iris', 'windflower', 'tree poppy',
    'gazania', 'azalea', 'water lily', 'rose', 'thorn apple', 'morning glory',
    'passion flower', 'lotus', 'toad lily', 'anthurium', 'frangipani', 'clematis',
    'hibiscus', 'columbine', 'desert-rose', 'tree mallow', 'magnolia', 'cyclamen',
    'watercress', 'canna lily', 'hippeastrum', 'bee balm', 'pink quill',
    'foxglove', 'bougainvillea', 'camellia', 'mallow', 'mexican petunia',
    'bromelia', 'blanket flower', 'trumpet creeper', 'blackberry lily',
]

TARGET_FLOWERS = [
    'Sunflower', 'Rose', 'Lotus', 'Hibiscus', 'Daffodil',
    'Magnolia', 'Azalea', 'Carnation', 'Petunia', 'Marigold',
    'Tiger Lily', 'Water Lily', 'Corn Poppy', 'Lavender', 'Tulip',
    'Bougainvillea', 'Camellia', 'Foxglove', 'Geranium', 'Clematis',
]


def load_oxford_flowers_dataset(num_images: int = 100,
                                 data_root: str = '/content/datasets/oxford_flowers102'
                                 ) -> list:
    print('📥 Loading Oxford Flowers 102 dataset…')
    images_dir = os.path.join(data_root, 'jpg')

    if not os.path.exists(images_dir):
        print('   ⚠ Not found — downloading…')
        os.makedirs(data_root, exist_ok=True)
        url = 'https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz'
        tgz = os.path.join(data_root, '102flowers.tgz')
        try:
            urllib.request.urlretrieve(url, tgz)
            print('   ✓ Downloaded')
            with tarfile.open(tgz, 'r:gz') as tar:
                tar.extractall(data_root)
            os.remove(tgz)
            print('   ✓ Extracted')
        except Exception as e:
            raise RuntimeError(f'Failed to download Oxford Flowers 102: {e}')

    if not os.path.exists(images_dir):
        raise RuntimeError(f'Image directory not found at {images_dir}.')

    all_jpg = sorted([
        os.path.join(images_dir, f)
        for f in os.listdir(images_dir) if f.lower().endswith('.jpg')
    ])
    print(f'   ✓ Found {len(all_jpg)} flower images')

    images_per_class = max(1, len(all_jpg) // 102)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    np.random.seed(2024)
    indices = np.random.choice(len(all_jpg), min(num_images, len(all_jpg)),
                               replace=False)
    images_data = []
    for idx in indices:
        img_path = all_jpg[int(idx)]
        try:
            img_pil       = Image.open(img_path).convert('RGB')
            original_size = img_pil.size
            img_np        = transform(img_pil).permute(1, 2, 0).numpy()
            file_num      = int(
                os.path.splitext(os.path.basename(img_path))[0]
                .replace('image_', ''))
            cat_idx    = min((file_num - 1) // images_per_class, 101)
            class_name = OXFORD_FLOWERS_102_CATEGORIES[cat_idx].title()
            images_data.append({
                'array':         img_np,
                'path':          img_path,
                'original_size': original_size,
                'class_name':    class_name,
                'label':         cat_idx,
            })
        except Exception as e:
            print(f'   ⚠ Skipping {img_path}: {e}')

    cats = len(set(d['class_name'] for d in images_data))
    print(f'   ✓ Loaded {len(images_data)} images  ({cats} categories)')
    return images_data


# =============================================================================
# OFFLINE PHASE  (with LinUCB Contextual MAB in Step 7)
# =============================================================================
def offline_phase(model, extractor, images_data, device,
                  combo_library, fc_descs, n_cores,
                  all_energy_mJ, output_dir, steps=5):
    os.makedirs(output_dir, exist_ok=True)
    N             = len(images_data)
    kernels_cache = {}

    # ─── STEP 1 ──────────────────────────────────────────────────────────
    print('\n' + '='*70)
    print(f'OFFLINE STEP 1 — Detect {n_cores} TPU Cores  |  Select {n_cores} Combos')
    print('='*70)

    combo_energies   = {name: combo_energy_mJ(clist, fc_descs, steps)
                        for name, clist in combo_library.items()}
    sorted_by_energy = sorted(combo_energies, key=combo_energies.get)
    step_size        = max(1, len(sorted_by_energy) // n_cores)
    seen, selected_combos = set(), []
    for i in range(n_cores):
        c = sorted_by_energy[i * step_size]
        if c not in seen:
            seen.add(c); selected_combos.append(c)
    for c in sorted_by_energy:
        if len(selected_combos) >= n_cores:
            break
        if c not in seen:
            seen.add(c); selected_combos.append(c)
    selected_combos = selected_combos[:n_cores]

    ce    = {name: combo_energies[name] for name in selected_combos}
    E_max = max(ce.values()); E_min = min(ce.values())
    e_eff = {name: (E_max - ce[name]) / (E_max - E_min + 1e-12)
             for name in selected_combos}

    print(f"  {'Core':<8} {'Combo':<45} {'Level(s)':<12} {'Energy(mJ)':<14} {'E_eff'}")
    print('  ' + '-'*95)
    for ci, cname in enumerate(selected_combos):
        clist  = combo_library[cname]
        levels = '/'.join(str(MAC_CONFIGURATIONS[c]['level']) for c in clist)
        print(f'  Core {ci+1:<4} {cname:<45} {levels:<12} '
              f'{ce[cname]:<14.6f} {e_eff[cname]:.4f}')

    # ─── STEP 2 ──────────────────────────────────────────────────────────
    print(f'\nOFFLINE STEP 2 — Core → Combo Assignment')
    for k, cname in enumerate(selected_combos):
        print(f'  Core {k+1} → {cname}')

    # ─── STEP 4: Feature extraction  (BUG FIX) ───────────────────────────
    print(f'\nOFFLINE STEP 4 — Feature Extraction')

    extractor.eval()
    penult_features = []
    for i, img_data in enumerate(images_data):
        img_t = (torch.from_numpy(img_data['array'])
                 .permute(2, 0, 1).unsqueeze(0).float())
        with torch.no_grad():
            feat = extractor(img_t).squeeze(0).cpu().numpy()
        penult_features.append(feat)
        if (i + 1) % 20 == 0:
            print(f'  [{i+1}/{N}] penultimate features extracted')
    penult_features = np.array(penult_features, dtype=np.float32)

    # BUG FIX: clamp n_pca to valid range
    n_pca = min(PCA_COMPONENTS, N - 1, penult_features.shape[1])
    if n_pca < PCA_COMPONENTS:
        print(f'  ⚠ PCA_COMPONENTS clamped {PCA_COMPONENTS} → {n_pca}  '
              f'(requires n_pca ≤ min(N-1, n_features) = min({N-1}, {penult_features.shape[1]}))')
    print(f'  Penultimate layer: {penult_features.shape[1]}-D → PCA {n_pca}-D  '
          f'+  5 visual features')

    print(f'  Applying PCA: {penult_features.shape[1]}-D → {n_pca}-D')
    pca_scaler = StandardScaler()
    pf_scaled  = pca_scaler.fit_transform(penult_features)
    pca_model  = PCA(n_components=n_pca, random_state=42)
    pf_pca     = pca_model.fit_transform(pf_scaled)
    print(f'  PCA explained variance: '
          f'{pca_model.explained_variance_ratio_.sum()*100:.1f}%')

    print(f'  Extracting visual features…')
    vis_features = np.array([extract_visual_features(d['array'])
                              for d in images_data], dtype=np.float32)
    vis_scaler   = StandardScaler()
    vis_scaled   = vis_scaler.fit_transform(vis_features)

    enriched = np.concatenate([pf_pca, vis_scaled], axis=1)
    d_ctx    = enriched.shape[1]
    print(f'  Enriched feature vector shape: {enriched.shape}  '
          f'({n_pca} PCA + 5 visual)')

    image_feature_map = {i: {'class_name': images_data[i].get('class_name', '?'),
                              'penult_idx': i}
                         for i in range(N)}

    # ─── STEP 5: K-Means clustering ─────────────────────────────────────
    print(f'\nOFFLINE STEP 5 — K-Means Clustering  (K={n_cores})')
    K_actual = min(n_cores, N)
    if K_actual < n_cores:
        print(f'  ⚠ Only {N} images — reducing K to {K_actual}')

    kmeans         = KMeans(n_clusters=K_actual, random_state=42, n_init=20)
    cluster_labels = kmeans.fit_predict(enriched)
    centroids      = kmeans.cluster_centers_
    image_cluster_map = {i: int(cluster_labels[i]) for i in range(N)}

    print(f'  {K_actual} clusters formed')
    for k in range(K_actual):
        idxs    = [i for i in range(N) if cluster_labels[i] == k]
        samples = [images_data[i].get('class_name', '?') for i in idxs]
        print(f'  Cluster {k} ({len(idxs)} images): '
              f'{samples[:5]}{"..." if len(samples) > 5 else ""}')

    # ─── STEP 6: Accurate gradient maps ─────────────────────────────────
    print(f'\nOFFLINE STEP 6 — Accurate Gradient Maps  (exact multiplier L0)')
    accurate_gmaps = []
    for i, img_data in enumerate(images_data):
        img_t    = torch.from_numpy(img_data['array']).float().to(device)
        baseline = torch.zeros_like(img_t)
        gmap     = compute_integrated_gradients_accurate(
            model, img_t, baseline, steps, device)
        accurate_gmaps.append(gmap)
        print(f'  [{i+1}/{N}] {img_data.get("class_name","?")}')
        del img_t, baseline
        gc.collect()
    print(f'  ✓ {N} accurate gradient maps computed')

    # ─── STEP 7: LinUCB CONTEXTUAL MAB ───────────────────────────────────
    print(f'\nOFFLINE STEP 7 — LinUCB Contextual Multi-Armed Bandit')
    print(f'  Arms      : {len(selected_combos)} selected combos')
    print(f'  Context   : cluster centroid  (d={d_ctx})')
    print(f'  α         : {LINUCB_ALPHA}')
    print(f'  Rounds    : {LINUCB_ROUNDS} pass(es) over {N} training images')
    print(f'  Reward R  : {REWARD_WEIGHTS["energy_efficiency"]}·E_eff  '
          f'+ {REWARD_WEIGHTS["spearman_correlation"]}·Spearman  '
          f'+ {REWARD_WEIGHTS["top_pixel_concentration"]}·TPC  '
          f'(Spearman gate ≥ {SPEARMAN_GATE_FRACTION}·best)')

    bandit = LinUCBBandit(
        n_arms      = len(selected_combos),
        d           = d_ctx,
        alpha       = LINUCB_ALPHA,
        combo_names = selected_combos,
    )

    # ── Seed phase ────────────────────────────────────────────────────────
    print(f'\n  [Seed phase] Each arm evaluated once per cluster '
          f'({len(selected_combos)} arms × {K_actual} clusters)…')
    seed_spearman = {}

    for ci, cname in enumerate(selected_combos):
        clist = combo_library[cname]
        for k in range(K_actual):
            cluster_idxs = [i for i in range(N) if cluster_labels[i] == k]
            if not cluster_idxs:
                continue
            rep_i = cluster_idxs[0]
            ctx   = centroids[k]

            img_t    = torch.from_numpy(images_data[rep_i]['array']).float().to(device)
            baseline = torch.zeros_like(img_t)
            apx_gmap = compute_integrated_gradients_combo(
                model, img_t, baseline, steps, clist, device, kernels_cache)
            del img_t, baseline; gc.collect()

            spear = calculate_spearman_correlation(accurate_gmaps[rep_i], apx_gmap)
            tpc   = calculate_top_pixel_concentration(apx_gmap)
            seed_spearman[(k, ci)] = spear

            R = (REWARD_WEIGHTS['energy_efficiency']    * e_eff[cname]
               + REWARD_WEIGHTS['spearman_correlation'] * spear
               + REWARD_WEIGHTS['top_pixel_concentration'] * tpc)

            bandit.update(ci, ctx, R)
            print(f'    seed  k={k}  arm={ci} ({cname[:28]:<28})  '
                  f'R={R:.4f}  ρ={spear:.4f}')

    cluster_best_spearman = {}
    for k in range(K_actual):
        vals = [seed_spearman[(k, ci)]
                for ci in range(len(selected_combos)) if (k, ci) in seed_spearman]
        cluster_best_spearman[k] = max(vals) if vals else 1.0

    # ── LinUCB rounds ─────────────────────────────────────────────────────
    print(f'\n  [LinUCB rounds] {LINUCB_ROUNDS} pass(es) × {N} images…')
    reward_log = []

    for rnd in range(LINUCB_ROUNDS):
        print(f'    — Round {rnd + 1}/{LINUCB_ROUNDS}')
        for i in range(N):
            k   = int(cluster_labels[i])
            ctx = centroids[k]

            arm_idx = bandit.select(ctx)
            cname   = selected_combos[arm_idx]
            clist   = combo_library[cname]

            img_t    = torch.from_numpy(images_data[i]['array']).float().to(device)
            baseline = torch.zeros_like(img_t)
            apx_gmap = compute_integrated_gradients_combo(
                model, img_t, baseline, steps, clist, device, kernels_cache)
            del img_t, baseline; gc.collect()

            spear = calculate_spearman_correlation(accurate_gmaps[i], apx_gmap)
            tpc   = calculate_top_pixel_concentration(apx_gmap)

            gated_ee = (e_eff[cname]
                        if spear >= SPEARMAN_GATE_FRACTION * cluster_best_spearman[k]
                        else 0.0)
            R = (REWARD_WEIGHTS['energy_efficiency']    * gated_ee
               + REWARD_WEIGHTS['spearman_correlation'] * spear
               + REWARD_WEIGHTS['top_pixel_concentration'] * tpc)

            bandit.update(arm_idx, ctx, R)
            reward_log.append((rnd + 1, i, k, arm_idx, R, spear))
            cluster_best_spearman[k] = max(cluster_best_spearman[k], spear)

    # ── Final mapping ─────────────────────────────────────────────────────
    cluster_to_combo = {}
    print(f'\n  LinUCB final cluster→combo (argmax exploitation score):')
    print(f"  {'Cluster':<10} {'Best Combo':<45} {'Exploit Score':<15} {'Level(s)'}")
    print('  ' + '-'*90)
    for k in range(K_actual):
        ctx     = centroids[k]
        best_ai = bandit.best_arm(ctx)
        cname   = selected_combos[best_ai]
        score   = bandit.exploit_score(best_ai, ctx)
        cluster_to_combo[k] = cname
        clist   = combo_library[cname]
        levels  = '/'.join(str(MAC_CONFIGURATIONS[c]['level']) for c in clist)
        print(f'  Cluster {k:<4} {cname:<45} {score:<15.4f} {levels}')

    bandit.summary(centroids)

    n_sel         = len(selected_combos)
    reward_matrix = np.zeros((N, n_sel), dtype=np.float32)
    for _, i, _, arm_idx, R, _ in reward_log:
        reward_matrix[i, arm_idx] = R

    # ─── STEP 8: Visualisations ───────────────────────────────────────────
    print(f'\nOFFLINE STEP 8 — Gradient Map Visualisations for Representative Flowers')
    gmap_dir = os.path.join(output_dir, 'representative_gradient_maps')
    os.makedirs(gmap_dir, exist_ok=True)

    for target_flower in TARGET_FLOWERS:
        match_idx = next(
            (i for i, d in enumerate(images_data)
             if d.get('class_name', '').lower() == target_flower.lower()),
            None)
        if match_idx is None:
            print(f'  ⚠ "{target_flower}" not found — skipping')
            continue

        img_data  = images_data[match_idx]
        flower_tag= target_flower.replace(' ', '_')
        orig_sz   = img_data.get('original_size', (224, 224))
        img_path  = img_data.get('path', '')
        cluster_k = int(cluster_labels[match_idx])
        combo_k   = cluster_to_combo[cluster_k]
        clist_k   = combo_library[combo_k]
        mult_k    = MAC_CONFIGURATIONS[clist_k[0]]['multiplier']

        print(f'  {target_flower}  (img{match_idx}, cluster={cluster_k}, '
              f'combo={combo_k})')

        flower_dir = os.path.join(gmap_dir, flower_tag)
        os.makedirs(flower_dir, exist_ok=True)

        if os.path.exists(str(img_path)):
            Image.open(img_path).convert('RGB').save(
                os.path.join(flower_dir, f'{flower_tag}_original.png'))

        img_t    = torch.from_numpy(img_data['array']).float().to(device)
        baseline = torch.zeros_like(img_t)

        acc_gmap = compute_integrated_gradients_accurate(
            model, img_t, baseline, steps, device)
        cv2.imwrite(os.path.join(flower_dir, f'{flower_tag}_ACCURATE_gray.png'),
                    create_high_res_red_gradient_map(acc_gmap, orig_sz, apply_red=False))
        cv2.imwrite(os.path.join(flower_dir, f'{flower_tag}_ACCURATE_red.png'),
                    create_high_res_red_gradient_map(acc_gmap, orig_sz, apply_red=True))
        jet_acc = create_jet_colormap_gradient(acc_gmap, orig_sz)
        cv2.imwrite(os.path.join(flower_dir, f'{flower_tag}_ACCURATE_jet.png'),
                    cv2.cvtColor(jet_acc, cv2.COLOR_RGB2BGR))

        apx_gmap = compute_integrated_gradients_combo(
            model, img_t, baseline, steps, clist_k, device, kernels_cache)
        cv2.imwrite(os.path.join(flower_dir,
                    f'{flower_tag}_{mult_k}_gray.png'),
                    create_high_res_red_gradient_map(apx_gmap, orig_sz, apply_red=False))
        cv2.imwrite(os.path.join(flower_dir,
                    f'{flower_tag}_{mult_k}_red.png'),
                    create_high_res_red_gradient_map(apx_gmap, orig_sz, apply_red=True))
        jet_apx = create_jet_colormap_gradient(apx_gmap, orig_sz)
        cv2.imwrite(os.path.join(flower_dir,
                    f'{flower_tag}_{mult_k}_jet.png'),
                    cv2.cvtColor(jet_apx, cv2.COLOR_RGB2BGR))

        if os.path.exists(str(img_path)):
            fig_path = save_gradient_comparison_figure(
                img_path, orig_sz, acc_gmap, apx_gmap,
                clist_k[0], target_flower, match_idx,
                flower_dir, flower_tag)
            if fig_path:
                print(f'    Comparison figure → {os.path.basename(fig_path)}')

        spear = calculate_spearman_correlation(acc_gmap, apx_gmap)
        print(f'    Spearman(acc vs assigned) = {spear:.4f}  → {flower_dir}')
        del img_t, baseline; gc.collect()

    print(f'  ✓ Gradient maps saved → {gmap_dir}')

    print(f'\nOFFLINE STEP 8 — Finalise Mapping  (LinUCB-derived)')
    print(f"  {'Cluster/Core':<14} {'Best Combo':<45} {'Level(s)'}")
    print('  ' + '-'*75)
    for k in range(K_actual):
        cname  = cluster_to_combo[k]
        clist  = combo_library[cname]
        levels = '/'.join(str(MAC_CONFIGURATIONS[c]['level']) for c in clist)
        print(f'  Core {k+1:<10} {cname:<45} {levels}')

    mapping = {
        'cluster_to_combo':   cluster_to_combo,
        'combo_library':      combo_library,
        'selected_combos':    selected_combos,
        'centroids':          centroids,
        'pca_model':          pca_model,
        'pca_scaler':         pca_scaler,
        'vis_scaler':         vis_scaler,
        'cluster_labels':     cluster_labels,
        'image_cluster_map':  image_cluster_map,
        'image_feature_map':  image_feature_map,
        'K':                  K_actual,
        'reward_matrix':      reward_matrix,
        'n_cores':            n_cores,
        'linucb_A':           [a.copy() for a in bandit.A],
        'linucb_b':           [b.copy() for b in bandit.b],
        'linucb_pulls':       bandit.pull_counts.copy(),
        'linucb_alpha':       LINUCB_ALPHA,
    }
    save_path = os.path.join(output_dir, 'cluster_combo_mapping.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump(mapping, f)
    print(f'  ✓ Mapping + LinUCB state saved → {save_path}')
    return mapping


# =============================================================================
# ONLINE PHASE  (pure deterministic lookup — no RL at runtime)
# =============================================================================
def online_phase(model, extractor, img_data, mapping, device,
                 output_dir, steps=5, img_idx=0):
    os.makedirs(output_dir, exist_ok=True)

    combo_library    = mapping['combo_library']
    centroids        = mapping['centroids']
    pca_model        = mapping['pca_model']
    pca_scaler       = mapping['pca_scaler']
    vis_scaler       = mapping['vis_scaler']
    cluster_to_combo = mapping['cluster_to_combo']
    kernels_cache    = {}

    class_name = img_data.get('class_name', 'Unknown')
    img_arr    = img_data['array']
    orig_sz    = img_data.get('original_size', (224, 224))
    img_path   = img_data.get('path', '')

    print(f"\n{'='*60}")
    print(f'ONLINE PHASE — Image {img_idx}: {class_name}')
    print(f"{'='*60}")

    # ─── STEP 1 ──────────────────────────────────────────────────────────
    extractor.eval()
    img_t = (torch.from_numpy(img_arr)
             .permute(2, 0, 1).unsqueeze(0).float())
    with torch.no_grad():
        penult = extractor(img_t).squeeze(0).cpu().numpy()

    penult_scaled = pca_scaler.transform(penult.reshape(1, -1))
    penult_pca    = pca_model.transform(penult_scaled)

    vis   = extract_visual_features(img_arr).reshape(1, -1)
    vis_s = vis_scaler.transform(vis)

    enriched = np.concatenate([penult_pca, vis_s], axis=1)
    print(f'  ✓ Enriched feature vector: {enriched.shape}')

    # ─── STEP 2 ──────────────────────────────────────────────────────────
    dists           = np.linalg.norm(centroids - enriched, axis=1)
    nearest_cluster = int(np.argmin(dists))
    print(f'  Distances to centroids: {np.round(dists, 4)}')
    print(f'  → Nearest cluster: {nearest_cluster}')

    # ─── STEP 3 ──────────────────────────────────────────────────────────
    combo_name = cluster_to_combo[nearest_cluster]
    combo_list = combo_library[combo_name]
    mult_name  = MAC_CONFIGURATIONS[combo_list[0]]['multiplier']
    print(f'  → Assigned combo : {combo_name}  [LinUCB offline decision]')
    print(f'  → Routed to      : TPU Core {nearest_cluster + 1}')
    for li, cfg in enumerate(combo_list):
        print(f'      FC{li}: {MAC_CONFIGURATIONS[cfg]["multiplier"]}  '
              f'(L{MAC_CONFIGURATIONS[cfg]["level"]})')

    # ─── STEP 4 ──────────────────────────────────────────────────────────
    img_tensor = torch.from_numpy(img_arr).float().to(device)
    baseline   = torch.zeros_like(img_tensor)

    print(f'  Computing approximate gradient map…')
    apx_gmap = compute_integrated_gradients_combo(
        model, img_tensor, baseline, steps, combo_list, device, kernels_cache)

    print(f'  Computing accurate gradient map (reference)…')
    acc_gmap = compute_integrated_gradients_accurate(
        model, img_tensor, baseline, steps, device)

    del img_tensor, baseline; gc.collect()

    # ─── STEP 5 ──────────────────────────────────────────────────────────
    tag = class_name.replace(' ', '_')

    cv2.imwrite(os.path.join(output_dir,
                f'online_img{img_idx}_{tag}_approx_gray.png'),
                create_high_res_red_gradient_map(apx_gmap, orig_sz, apply_red=False))
    cv2.imwrite(os.path.join(output_dir,
                f'online_img{img_idx}_{tag}_approx_red.png'),
                create_high_res_red_gradient_map(apx_gmap, orig_sz, apply_red=True))
    jet_apx = create_jet_colormap_gradient(apx_gmap, orig_sz)
    cv2.imwrite(os.path.join(output_dir,
                f'online_img{img_idx}_{tag}_approx_jet.png'),
                cv2.cvtColor(jet_apx, cv2.COLOR_RGB2BGR))

    cv2.imwrite(os.path.join(output_dir,
                f'online_img{img_idx}_{tag}_accurate_gray.png'),
                create_high_res_red_gradient_map(acc_gmap, orig_sz, apply_red=False))
    cv2.imwrite(os.path.join(output_dir,
                f'online_img{img_idx}_{tag}_accurate_red.png'),
                create_high_res_red_gradient_map(acc_gmap, orig_sz, apply_red=True))
    jet_acc = create_jet_colormap_gradient(acc_gmap, orig_sz)
    cv2.imwrite(os.path.join(output_dir,
                f'online_img{img_idx}_{tag}_accurate_jet.png'),
                cv2.cvtColor(jet_acc, cv2.COLOR_RGB2BGR))

    if os.path.exists(str(img_path)):
        Image.open(img_path).convert('RGB').save(
            os.path.join(output_dir,
                         f'online_img{img_idx}_{tag}_original.png'))

        fig_path = save_gradient_comparison_figure(
            img_path, orig_sz, acc_gmap, apx_gmap,
            combo_list[0], class_name, img_idx,
            output_dir, f'online_img{img_idx}_{tag}')
        if fig_path:
            print(f'  ✓ Comparison figure → {os.path.basename(fig_path)}')

        save_accurate_only_figure(
            img_path, orig_sz, acc_gmap, class_name, img_idx,
            output_dir, f'online_img{img_idx}_{tag}')

    print(f'  ✓ Gradient maps saved  [Core {nearest_cluster + 1}]')

    return {
        'class_name':      class_name,
        'nearest_cluster': nearest_cluster,
        'combo_name':      combo_name,
        'combo_list':      combo_list,
    }


# =============================================================================
# MAIN
# =============================================================================
if __name__ == '__main__':

    src_kernel_dir  = '/content/drive/MyDrive/adapt-main/adapt/cpu-kernels'
    dst_kernel_dir  = '/content/adapt/cpu-kernels'
    checkpoint_path = '/content/mobilenetv2_checkpoint.pth'
    data_root       = '/content/datasets/oxford_flowers102'
    offline_out     = '/content/tpu_offline'
    online_out      = '/content/tpu_online'

    print('\n' + '='*70)
    print('DESIGN-TIME APPROXIMATE COMPUTING FRAMEWORK')
    print('MobileNetV2 — Oxford Flowers 102 — Heterogeneous Combo Assignment')
    print(f'  P = {N_CORES} TPU cores  |  All 11 EvoApprox multipliers')
    print(f'  RL: LinUCB Contextual MAB  (offline Step 7 only)')
    print('='*70)

    # ── Copy C++ kernel files ────────────────────────────────────────────
    os.makedirs(f'{dst_kernel_dir}/axx_mults', exist_ok=True)
    print('\n📋 Copying C++ kernel files…')
    try:
        shutil.copy(f'{src_kernel_dir}/axx_linear.cpp',
                    f'{dst_kernel_dir}/axx_linear.cpp')
    except FileNotFoundError as e:
        print(f'  ✗ {e}')

    missing = []
    for cfg in MAC_CONFIGURATIONS.values():
        m   = cfg['multiplier']
        src = f'{src_kernel_dir}/axx_mults/{m}.h'
        dst = f'{dst_kernel_dir}/axx_mults/{m}.h'
        if os.path.exists(src):
            shutil.copy(src, dst)
        else:
            missing.append(m)

    if missing:
        print(f'  ⚠ Missing multiplier headers: {missing}')
        for k in list(MAC_CONFIGURATIONS.keys()):
            if MAC_CONFIGURATIONS[k]['multiplier'] in missing:
                del MAC_CONFIGURATIONS[k]
    print(f'  ✓ {len(MAC_CONFIGURATIONS)} multipliers available')

    P = N_CORES
    print(f'\n🔧 OFFLINE STEP 1 — Detect {P} TPU Cores')
    print(f'  P = {P} (set manually to match hardware)')

    device      = torch.device('cpu')
    num_classes = 102
    print(f'\n📦 Loading MobileNetV2 ({num_classes} classes — Oxford Flowers 102)…')

    checkpoint = None
    if os.path.exists(checkpoint_path):
        try:
            checkpoint = torch.load(checkpoint_path, map_location='cpu')
            state      = checkpoint.get('model_state_dict', checkpoint)
            for key in ['model.classifier.1.weight', 'classifier.1.weight']:
                if key in state:
                    num_classes = state[key].shape[0]
                    print(f'  ℹ Detected num_classes={num_classes} from "{key}"')
                    break
        except Exception as e:
            print(f'  ⚠ Could not read checkpoint: {e}')
            checkpoint = None
    else:
        print(f'  ⚠ Checkpoint not found at {checkpoint_path} — using random weights')

    model = MobileNetV2Classifier(num_classes=num_classes)
    if checkpoint is not None:
        try:
            model.load_state_dict(
                checkpoint.get('model_state_dict', checkpoint), strict=False)
            print(f'  ✓ Loaded | Epoch {checkpoint.get("epoch","?")} '
                  f'| Acc {checkpoint.get("test_acc", 0):.2f}%')
        except Exception as e:
            print(f'  ⚠ Partial load: {e}')
    else:
        print('  ✓ Using random initialisation')

    model.to(device).eval()
    extractor = PenultimateExtractor(model).to(device).eval()

    profiler          = LayerProfiler(model, input_shape=(1, 3, 224, 224))
    layer_descriptors = profiler.profile()
    fc_descs          = [d for d in layer_descriptors if d.layer_type == 'fc']
    n_fc_layers       = len(fc_descs)
    print(f'  ✓ {len(layer_descriptors)} layers profiled  ({n_fc_layers} FC layer(s))')

    all_energy_mJ = {}
    for cfg_name in MAC_CONFIGURATIONS:
        em = RoHNASEnergyModel(cfg_name)
        all_energy_mJ[cfg_name] = em.calculate_ig_energy(
            5, layer_descriptors)['ig_energy_mJ']
    E_max_all = max(all_energy_mJ.values())
    E_min_all = min(all_energy_mJ.values())
    print(f'  E_max (all configs) = {E_max_all:.6f} mJ')
    print(f'  E_min (all configs) = {E_min_all:.6f} mJ')

    print(f'\n📐 Building combo library  ({n_fc_layers} FC layer(s) per combo)…')
    combo_library = build_combo_library(n_fc_layers)

    print(f'\n📸 OFFLINE STEP 3 — Collect Training Images (Oxford Flowers 102)')
    images_data = load_oxford_flowers_dataset(num_images=100, data_root=data_root)
    cat_counts  = {}
    for d in images_data:
        cat_counts[d['class_name']] = cat_counts.get(d['class_name'], 0) + 1
    print(f'  Categories ({len(cat_counts)} unique): {sorted(cat_counts.keys())}')

    # ── OFFLINE PHASE ────────────────────────────────────────────────────
    mapping = offline_phase(
        model         = model,
        extractor     = extractor,
        images_data   = images_data,
        device        = device,
        combo_library = combo_library,
        fc_descs      = fc_descs,
        n_cores       = N_CORES,
        all_energy_mJ = all_energy_mJ,
        output_dir    = offline_out,
        steps         = 5,
    )

    # ── ONLINE PHASE ─────────────────────────────────────────────────────
    print('\n\n' + '='*70)
    print('ONLINE PHASE — Runtime Routing  (deterministic, no RL)')
    print('Routing: enriched features → nearest centroid → LinUCB combo lookup')
    print('='*70)

    np.random.seed(999)
    online_pool  = load_oxford_flowers_dataset(num_images=20, data_root=data_root)
    train_paths  = {d['path'] for d in images_data}
    online_pool  = [d for d in online_pool if d['path'] not in train_paths]
    online_sample= online_pool[:3] if len(online_pool) >= 3 else online_pool

    online_results = []
    for i, img_entry in enumerate(online_sample):
        result = online_phase(
            model      = model,
            extractor  = extractor,
            img_data   = img_entry,
            mapping    = mapping,
            device     = device,
            output_dir = online_out,
            steps      = 5,
            img_idx    = i,
        )
        online_results.append(result)
        gc.collect()

    # ── Final summary ────────────────────────────────────────────────────
    K = mapping['K']
    print('\n\n' + '='*70)
    print('FINAL CLUSTER → COMBO ASSIGNMENT')
    print('(Derived by LinUCB offline — argmax exploitation score per centroid)')
    print('='*70)
    print(f"  {'Cluster/Core':<14} {'Best Combo':<45} {'Level(s)'}")
    print('  ' + '-'*75)
    for k in range(K):
        cname  = mapping['cluster_to_combo'].get(k, mapping['selected_combos'][0])
        clist  = combo_library[cname]
        levels = '/'.join(str(MAC_CONFIGURATIONS[c]['level']) for c in clist)
        print(f'  Core {k+1:<10} {cname:<45} {levels}')

    print(f'\n  Online routing results:')
    print(f"  {'Flower':<25} {'Core':<8} {'Assigned Combo'}")
    print('  ' + '-'*80)
    for r in online_results:
        print(f"  {r['class_name']:<25} Core {r['nearest_cluster']+1:<4} "
              f"{r['combo_name']}")

    print(f'\n✅ COMPLETE')
    print(f'  Offline outputs → {offline_out}')
    print(f'  Online outputs  → {online_out}')
    print('='*70)


DESIGN-TIME APPROXIMATE COMPUTING FRAMEWORK
MobileNetV2 — Oxford Flowers 102 — Heterogeneous Combo Assignment
  P = 5 TPU cores  |  All 11 EvoApprox multipliers
  RL: LinUCB Contextual MAB  (offline Step 7 only)

📋 Copying C++ kernel files…
  ✓ 11 multipliers available

🔧 OFFLINE STEP 1 — Detect 5 TPU Cores
  P = 5 (set manually to match hardware)

📦 Loading MobileNetV2 (102 classes — Oxford Flowers 102)…
  ℹ Detected num_classes=102 from "model.classifier.1.weight"
  ✓ Loaded | Epoch 20 | Acc 68.62%
  ✓ 53 layers profiled  (1 FC layer(s))
  E_max (all configs) = 0.922337 mJ
  E_min (all configs) = 0.762144 mJ

📐 Building combo library  (1 FC layer(s) per combo)…
  Combo library: 120 heterogeneous combos  | FC layers per combo: 1  | Levels covered: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

📸 OFFLINE STEP 3 — Collect Training Images (Oxford Flowers 102)
📥 Loading Oxford Flowers 102 dataset…
   ✓ Found 8189 flower images
   ✓ Loaded 100 images  (67 categories)
  Categories (67 unique): ['Anth